# Detekcja Naczyń Krwionośnych Dna Oka
Wybierz metodę, obraz wejściowy i kliknij "Przetwórz Obraz". Po wygenerowaniu wyniku możesz zapisać maskę.

In [7]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import naszych modułów
from detect import VesselDetector
from przetwarzanie import VesselExtractor

In [8]:
# Ścieżki do folderów
INPUT_DIR = Path('./data/test/input')
OUTPUT_DIR = Path('./data/results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Wyszukaj dostępne obrazy
image_files = sorted([f.name for f in INPUT_DIR.glob('*.ppm')])
if not image_files:
    image_files = ['Brak obrazów']

# Zmienne globalne
generated_mask = None
current_image_name = None

# Elementy interfejsu (Widżety)
w_method = widgets.Dropdown(
    options=['Podstawowa (Frangi)', 'Uczenie Maszynowe (Random Forest)', 'Deep Learning (U-Net)'],
    value='Podstawowa (Frangi)',
    description='Metoda:',
    style={'description_width': 'initial'}
)

w_image = widgets.Dropdown(
    options=image_files,
    description='Obraz:',
    style={'description_width': 'initial'}
)

w_upload = widgets.FileUpload(
    accept='.ppm, .png, .jpg, .jpeg',
    multiple=False,
    description='Wgraj plik'
)

w_process_btn = widgets.Button(description='Przetwórz Obraz', button_style='primary')
w_save_btn = widgets.Button(description='Zapisz Maskę', button_style='success')
w_output = widgets.Output()

def handle_upload(change):
    if not change.new: return
    uploaded = change.new
    
    # Wsparcie dla ipywidgets 7.x i 8.x
    if isinstance(uploaded, dict):
        for filename, file_info in uploaded.items():
            content = file_info['content']
            break
    else:
        file_info = uploaded[0]
        filename = file_info['name']
        content = file_info['content']
        
    try:
        with open(INPUT_DIR / filename, 'wb') as f:
            f.write(content)
        
        image_files = sorted([f.name for f in INPUT_DIR.glob('*') if f.suffix.lower() in ['.ppm', '.png', '.jpg', '.jpeg']])
        w_image.options = image_files
        w_image.value = filename
        with w_output:
            print(f"Wgrano plik {filename}. Możesz teraz go przetworzyć.")
    except Exception as e:
        with w_output:
            print(f"Błąd zapisu pliku: {e}")
            
    # Zresetowanie widżetu, by móc wgrać ponownie ten sam plik
    w_upload.value = () if isinstance(uploaded, (list, tuple)) else {}

w_upload.observe(handle_upload, names='value')

def process_image(b):
    global generated_mask, current_image_name
    w_process_btn.disabled = True
    with w_output:
        clear_output(wait=True)
        if w_image.value == 'Brak obrazów':
            print("Brak dostępnych obrazów do przetworzenia.")
            w_process_btn.disabled = False
            return
            
        print("Przetwarzanie w toku, proszę czekać...")
        
        image_path = INPUT_DIR / w_image.value
        current_image_name = w_image.value
        method = w_method.value
        
        try:
            if method == 'Podstawowa (Frangi)':
                detector = VesselDetector(model_path="basic_ip")
                generated_mask = detector.detect(image_path)
                
            elif method == 'Uczenie Maszynowe (Random Forest)':
                detector = VesselDetector(model_path="./model/rf_vessels.joblib")
                generated_mask = detector.detect(image_path)
                
            elif method == 'Deep Learning (U-Net)':
                detector = VesselDetector(model_path="./model/unet_vessels.pth")
                generated_mask = detector.detect(image_path)
                
            # Wyświetlanie wyników
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
            
            orig_img = Image.open(image_path)
            ax1.imshow(orig_img)
            ax1.set_title('Oryginalny Obraz')
            ax1.axis('off')
            
            ax2.imshow(generated_mask, cmap='gray')
            ax2.set_title(f'Wygenerowana Maska\n({method})')
            ax2.axis('off')
            
            plt.tight_layout()
            plt.show()
            plt.close(fig)
            
            print("Przetwarzanie zakończone pomyślnie!")
            
        except Exception as e:
            import traceback
            print(f"Wystąpił błąd podczas przetwarzania: {e}")
            traceback.print_exc()
        finally:
            w_process_btn.disabled = False

def save_mask(b):
    with w_output:
        if generated_mask is not None and current_image_name is not None:
            prefix = "podstawowa"
            if "Random Forest" in w_method.value: prefix = "rf"
            elif "U-Net" in w_method.value: prefix = "unet"
            
            save_name = f"{current_image_name.split('.')[0]}_{prefix}_mask.png"
            save_path = OUTPUT_DIR / save_name
            Image.fromarray(generated_mask).save(save_path)
            print(f"Zapisano maskę do: {save_path}")
        else:
            print("Najpierw przetwórz obraz, aby zapisać maskę.")

w_process_btn.on_click(process_image)
w_save_btn.on_click(save_mask)

# Budowa UI
ui = widgets.VBox([
    widgets.HTML("<h3>Wybierz parametry i przetwórz obraz</h3>"),
    widgets.HBox([w_method, w_image, w_upload], layout=widgets.Layout(padding='10px')),
    widgets.HBox([w_process_btn, w_save_btn], layout=widgets.Layout(padding='10px')),
    w_output
], layout=widgets.Layout(padding='20px'))

display(ui)